# Track B · VibeVoice-ASR 화자 구분 참고 행

화자 구분까지 하는 공개 파운데이션 ASR(`microsoft/VibeVoice-ASR`, MIT, 9B)이 **목소리만으로**
2인 평가셋을 얼마나 가르는지 잽니다. 이 모델은 마이크 한 개의 20초 전체를 듣고
`화자 · 시작–끝 시각 · 텍스트`를 내므로, 공간 정보를 쓰지 않고 인과 조건도 아닙니다.
그래서 리더보드에서는 공식(인과) 표 밖의 **참고 행**(`observation_mode = "full_audio"`)입니다.

이 노트북은 추론만 합니다. 각 세션에서 케이스가 듣는 채널(활성 마이크 중 가장 낮은 번호: `ch0`, 마스크 `0010`만 `ch2`)마다
한 번씩, `eval-t2`와 `eval-t2-static` 합계 800회를 돌려 결과를 Drive의 JSONL로 남깁니다.
채점은 로컬에서 `python3 track_b.py --segments <이 파일> --out runs/eval/track_b_vibevoice`로 합니다.

- 런타임: **A100 또는 L4** (bf16, 가중치만 약 18 GB). T4는 bf16을 지원하지 않습니다.
- 중간에 끊겨도 같은 셀을 다시 실행하면 끝난 세션은 건너뜁니다.

In [ ]:
BRANCH = "main"  # @param {type:"string"}
COMMIT = ""  # @param {type:"string"}
SETS = ["eval-t2", "eval-t2-static"]
BATCH = 4  # @param {type:"integer"}
MAX_NEW_TOKENS = 2048  # @param {type:"integer"}
OUT_NAME = "vibevoice_asr"  # @param {type:"string"}

REPO_URL = "https://github.com/hd1534/variable-channel-spatial-asr.git"
DRIVE_ROOT = "/content/drive/MyDrive/vcsasr"
REPO_DIR = "/content/variable-channel-spatial-asr"
MODEL_ID = "microsoft/VibeVoice-ASR"

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path


def sh(cmd, check=True):
    """셸 명령을 실행하고 출력을 셀에 바로 보여 준다. 실패하면 셀을 멈춘다."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0 and check:
        raise RuntimeError(f"실패 (종료 코드 {proc.returncode}): {cmd}")
    return proc.returncode


import torch

GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(f"torch {torch.__version__} | GPU: {GPU or '없음'}")
# is_bf16_supported() is True on a T4 too (emulation); native bf16 needs compute capability 8.x
if GPU is None or torch.cuda.get_device_capability()[0] < 8:
    raise RuntimeError("bf16을 지원하는 GPU(A100 또는 L4)가 필요합니다. 런타임 > 런타임 유형 변경에서 고르세요.")

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
Path(DRIVE_ROOT, "track_b").mkdir(parents=True, exist_ok=True)

if not Path(REPO_DIR, ".git").is_dir():
    # 평가셋 두 벌(약 1 GB)의 오디오는 받고, 단일 화자 eval-dev 오디오는 받지 않는다.
    sh(f'git clone -q --depth 1 --filter=blob:none --no-checkout -b {BRANCH} {REPO_URL} "{REPO_DIR}"')
    sh(f"git -C \"{REPO_DIR}\" sparse-checkout set --no-cone '/*' '!/data/eval-dev/audio/'")
sh(f'git -C "{REPO_DIR}" fetch -q --depth 1 origin {COMMIT or BRANCH} && git -C "{REPO_DIR}" checkout -q FETCH_HEAD')
os.chdir(REPO_DIR)
sh("git log -1 --format='%h %s'")

In [ ]:
# VibeVoice 공식 저장소의 패키지 (demo/vibevoice_asr_inference_from_file.py 와 같은 클래스를 쓴다)
if not Path("/content/VibeVoice").is_dir():
    sh("git clone -q --depth 1 https://github.com/microsoft/VibeVoice.git /content/VibeVoice")
sh("pip install -q -e /content/VibeVoice")
sh("git -C /content/VibeVoice log -1 --format='VibeVoice %h %ad' --date=short")

In [ ]:
from vibevoice.modular.modeling_vibevoice_asr import VibeVoiceASRForConditionalGeneration
from vibevoice.processor.vibevoice_asr_processor import VibeVoiceASRProcessor

try:
    import flash_attn  # noqa: F401
    ATTN = "flash_attention_2"
except ImportError:
    ATTN = "sdpa"
processor = VibeVoiceASRProcessor.from_pretrained(MODEL_ID, language_model_pretrained_name="Qwen/Qwen2.5-7B")
model = VibeVoiceASRForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, attn_implementation=ATTN, trust_remote_code=True).to("cuda").eval()
print("loaded", MODEL_ID, "| attention:", ATTN, f"| GPU memory {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
import soundfile as sf

# 케이스가 듣는 채널 = 활성 마이크 중 가장 낮은 번호 (다른 모든 행과 같은 규칙)
jobs = []
for s in SETS:
    seen = set()
    for line in open(f"data/{s}/meta.jsonl", encoding="utf-8"):
        m = json.loads(line)
        key = (s, m["session_id"], min(m["active_channels"]))
        if key not in seen:
            seen.add(key)
            jobs.append(key)

OUT = Path(DRIVE_ROOT, "track_b", f"{OUT_NAME}.jsonl")
done = set()
if OUT.exists():
    for line in OUT.read_text(encoding="utf-8").splitlines():
        if line.strip():
            r = json.loads(line)
            done.add((r["set"], r["session_id"], r["channel"]))
todo = [j for j in jobs if j not in done]
print(f"전체 {len(jobs)} · 끝남 {len(done)} · 남음 {len(todo)}")

TMP = Path("/content/track_b_wav")
TMP.mkdir(exist_ok=True)


def mono_wav(set_name, session_id, channel):
    path = TMP / f"{set_name}__{session_id}__ch{channel}.wav"
    if not path.exists():
        x, sr = sf.read(f"data/{set_name}/audio/{session_id}.wav", dtype="int16", always_2d=True)
        sf.write(path, x[:, channel], sr, subtype="PCM_16")
    return str(path)


t_start = time.time()
for i in range(0, len(todo), BATCH):
    batch = todo[i:i + BATCH]
    inputs = processor(audio=[mono_wav(*j) for j in batch], sampling_rate=None, return_tensors="pt", padding=True,
                       add_generation_prompt=True)
    inputs = {k: v.to("cuda") if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
    input_length = inputs["input_ids"].shape[1]
    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=processor.pad_id,
                                    eos_token_id=processor.tokenizer.eos_token_id, do_sample=False)
    sec = (time.time() - t0) / len(batch)
    rows = []
    for b, (s, sid, ch) in enumerate(batch):
        gen = output_ids[b, input_length:]
        eos = (gen == processor.tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
        if len(eos):
            gen = gen[: eos[0] + 1]
        text = processor.decode(gen, skip_special_tokens=True)
        try:
            segments = processor.post_process_transcription(text)
        except Exception as exc:  # 공식 데모와 같이: 구조화에 실패하면 빈 목록으로 남기고 원문은 보존
            print(f"  구조화 실패 {sid} ch{ch}: {exc}")
            segments = []
        rows.append({"set": s, "session_id": sid, "channel": ch, "backend": MODEL_ID, "raw_text": text,
                     "segments": [{k: seg.get(k) for k in ("start_time", "end_time", "speaker_id", "text")}
                                  for seg in segments],
                     "sec_per_session": round(sec, 3)})
    with open(OUT, "a", encoding="utf-8") as fh:  # 배치마다 Drive에 덧붙인다: 끊겨도 여기까지는 남는다
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    n = i + len(batch)
    if n % (BATCH * 10) == 0 or n == len(todo):
        eta = (time.time() - t_start) / n * (len(todo) - n)
        print(f"{n}/{len(todo)} · 세션당 {sec:.1f}s · 남은 시간 약 {eta / 60:.0f}분", flush=True)

In [ ]:
rows = [json.loads(l) for l in OUT.read_text(encoding="utf-8").splitlines() if l.strip()]
empty = [f"{r['set']}/{r['session_id']}/ch{r['channel']}" for r in rows if not r["segments"]]
n_spk = [len({s["speaker_id"] for s in r["segments"]}) for r in rows if r["segments"]]
print("===== VC-SASR TRACK B (paste this back) =====")
print(json.dumps({
    "file": str(OUT), "rows": len(rows), "expected": len(jobs), "empty_segments": len(empty),
    "empty_examples": empty[:5], "speakers_per_session": {str(k): n_spk.count(k) for k in sorted(set(n_spk))},
    "sec_per_session_mean": round(sum(r["sec_per_session"] for r in rows) / max(1, len(rows)), 2),
    "gpu": GPU, "example": rows[0] if rows else None,
}, ensure_ascii=False, indent=1))
print("===== END =====")
print("이 JSONL 파일을 내려받아 로컬에서: python3 track_b.py --segments <파일> --out runs/eval/track_b_vibevoice")